# Pipeline B → Pipeline A (v2)

Transformación del pipeline legacy (Google Sheets) a nuevo modelo.

Incluye:
- Normalización de contactos
- Expansión por múltiples contactos
- Extracción de `sales_owner` desde `Notes` (cuando sea posible)

Reglas:
- No se inventan datos
- `sales_owner` solo se asigna si hay señal clara
- `notes` NO se modifica

In [37]:
import pandas as pd

df_b = pd.read_csv("pipelineB.csv")

df_b.head()

,Brand,Last contact,Notes,Name,Email,Last Response,Followup Day,Status,First Contact
0,Estrella Damm : matheus will follow up,18/03/2026,wont continue,"Oriol, Team","otrullasb@damm.com, info@damm.es",18/03/2026,NaN,Rejected,3/03/2026
1,Mano Derecha,23/03/2026,Ya no trabajo en mano derecha si gustan ya no ...,Andres,andresberruetav@gmail.com,29/10/2026,NaN,Rejected,2/07/2025
2,De la rosa 613,13/04/2026,Getting the wines to the retailers was the iss...,"Yehudith, Andrew","delarosarealfoods@gmail.com, graphics@delarosa...",13/04/2026,NaN,Rejected,4/08/2025
3,Armed Forces Brewing,23/03/2026,We are contract brewing with thin margins. $35...,Alan,alan@armedforcesbrewingco.com,2/03/2026,NaN,Rejected,26/09/2025
4,Sogrape Spain,29/04/2026,hemos decidido no avanzar con la implementació...,"Trinidad, Julio, Alberto, Amaya, Lorea, Nacho","trinidad.villegas@bodegaslan.com, Julio.Martin...",28/04/2026,NaN,Rejected,24/02/2026


## Exploración inicial
Validamos estructura del pipeline B

In [38]:
df_b.info()
df_b.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 204 entries, 0 to 203
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Brand          204 non-null    object
 1   Last contact   204 non-null    object
 2   Notes          204 non-null    object
 3   Name           203 non-null    object
 4   Email          203 non-null    object
 5   Last Response  200 non-null    object
 6   Followup Day   54 non-null     object
 7   Status         204 non-null    object
 8   First Contact  79 non-null     object
dtypes: object(9)
memory usage: 14.5+ KB


,Brand,Last contact,Notes,Name,Email,Last Response,Followup Day,Status,First Contact
0,Estrella Damm : matheus will follow up,18/03/2026,wont continue,"Oriol, Team","otrullasb@damm.com, info@damm.es",18/03/2026,NaN,Rejected,3/03/2026
1,Mano Derecha,23/03/2026,Ya no trabajo en mano derecha si gustan ya no ...,Andres,andresberruetav@gmail.com,29/10/2026,NaN,Rejected,2/07/2025
2,De la rosa 613,13/04/2026,Getting the wines to the retailers was the iss...,"Yehudith, Andrew","delarosarealfoods@gmail.com, graphics@delarosa...",13/04/2026,NaN,Rejected,4/08/2025
3,Armed Forces Brewing,23/03/2026,We are contract brewing with thin margins. $35...,Alan,alan@armedforcesbrewingco.com,2/03/2026,NaN,Rejected,26/09/2025
4,Sogrape Spain,29/04/2026,hemos decidido no avanzar con la implementació...,"Trinidad, Julio, Alberto, Amaya, Lorea, Nacho","trinidad.villegas@bodegaslan.com, Julio.Martin...",28/04/2026,NaN,Rejected,24/02/2026
5,Gallo,2/03/2026,are sold through other fulfillment platforms ...,"Garrett, Tyler, Kim, Mark","garrett.fardelmann@ejgallo.com, Shana.Tyler@ej...",9/02/2026,NaN,Rejected,4/01/2025
6,Spencer Hoddeson (gay water),13/04/2026,asked for shipping rate and stopped responding,Spencer,spencer@gaywater.com,10/11/2026,NaN,Rejected,15/09/2025
7,RON GOBERNADOR 12YRS,26/01/2026,"Dejo de contestar el 24 de julio, ya tienen la...","Nicolas, Antonio, Homero","nicolasgonzm@gmail.com, antoniogonzalezposada@...",30/09/2025,NaN,Rejected,28/05/2025
8,Santo Spirits,20/01/2026,EMAIL CONTACT NO LONGER AVAILABLE,Ana,ana@santospirit.com,8/10/2025,NaN,Rejected,29/09/2025
9,Untamed Beverages,14/04/2026,first contact - At this point it looks like we...,"Joe, Lee, Bruce","joe.colella@untamedbeverages.com, lee.thaxton@...",14/04/2026,NaN,Rejected,6/04/2026


## Normalización de contactos

- Separar por coma
- Trim espacios
- Capitalizar nombres
- Mantener alineación nombre-email

In [39]:
def clean_name(name):
    if pd.isna(name):
        return None
    return name.strip().capitalize()

def split_and_clean(field):
    if pd.isna(field):
        return []
    return [item.strip() for item in field.split(",")]

## Extracción de sales_owner desde Notes

Reglas:

Detectamos patrones dentro de `Notes`:

### SofiG
- "sofi g"
- "sofig"
- "first onboarding call and email"

### Juan
- "juan"
- "first contact" (cuando explícitamente indica de Juan)

Si no hay match → None

IMPORTANTE:
- No modificamos `Notes`
- Matching case-insensitive

In [40]:
def extract_sales_owner(notes):
    if pd.isna(notes):
        return None
    
    text = notes.lower()
    
    # Reglas SofiG
    if "sofi g" in text or "sofig" in text:
        return "SofiG"
    if "first onboarding call and email" in text:
        return "SofiG"
    
    # Reglas Juan
    if "juan" in text:
        return "Juan"
    if "first contact" in text:
        return "Juan"
    
    return None

## Transformación completa

Incluye:
- Expansión por contactos
- Asignación de sales_owner
- Generación de contact_is_primary

In [41]:
rows = []

for _, row in df_b.iterrows():
    names = split_and_clean(row["Name"])
    emails = split_and_clean(row["Email"])
    
    sales_owner = extract_sales_owner(row["Notes"])
    
    max_len = min(len(names), len(emails))
    
    for i in range(max_len):
        new_row = {
            "brand_name": row["Brand"],
            "introduced_by": None,
            "sales_owner": sales_owner,
            "status": row["Status"],
            "follow_up_later_until": row["Followup Day"],
            "first_contact_at": row["First Contact"],
            "last_contact_at": row["Last contact"],
            "last_response_at": row["Last Response"],
            "notes": row["Notes"],
            
            "contact_full_name": clean_name(names[i]),
            "contact_email": emails[i] if pd.notna(emails[i]) else None,
            "contact_phone": None,
            "contact_role_title": None,
            "contact_is_primary": True if i == 0 else False
        }
        
        rows.append(new_row)

df_a = pd.DataFrame(rows)

In [42]:
from dateutil import parser

def clean_date(value):
    if pd.isna(value):
        return None
    
    value = str(value).strip()
    
    # Casos basura
    if value.lower() in ["no response", "nan", ""]:
        return None
    
    try:
        dt = parser.parse(value, dayfirst=True)  
        return dt.strftime("%Y-%m-%d")
    except:
        return None

In [ ]:
date_columns = [
    "First Contact",
    "Last contact",
    "Last Response",
    "Followup Day"
]
# limpieza fechas

for col in date_columns:
    df_b[col] = df_b[col].apply(clean_date)
    
df_b.head(10)

## Preview del resultado

Validamos:
- Expansión correcta
- sales_owner detectado correctamente
- primary contact correcto

In [44]:
df_a.head(20)

,brand_name,introduced_by,sales_owner,status,follow_up_later_until,first_contact_at,last_contact_at,last_response_at,notes,contact_full_name,contact_email,contact_phone,contact_role_title,contact_is_primary
0,Estrella Damm : matheus will follow up,None,None,Rejected,NaN,3/03/2026,18/03/2026,18/03/2026,wont continue,Oriol,otrullasb@damm.com,None,None,True
1,Estrella Damm : matheus will follow up,None,None,Rejected,NaN,3/03/2026,18/03/2026,18/03/2026,wont continue,Team,info@damm.es,None,None,False
2,Mano Derecha,None,None,Rejected,NaN,2/07/2025,23/03/2026,29/10/2026,Ya no trabajo en mano derecha si gustan ya no ...,Andres,andresberruetav@gmail.com,None,None,True
3,De la rosa 613,None,None,Rejected,NaN,4/08/2025,13/04/2026,13/04/2026,Getting the wines to the retailers was the iss...,Yehudith,delarosarealfoods@gmail.com,None,None,True
4,De la rosa 613,None,None,Rejected,NaN,4/08/2025,13/04/2026,13/04/2026,Getting the wines to the retailers was the iss...,Andrew,graphics@delarosa613.com,None,None,False
5,Armed Forces Brewing,None,None,Rejected,NaN,26/09/2025,23/03/2026,2/03/2026,We are contract brewing with thin margins. $35...,Alan,alan@armedforcesbrewingco.com,None,None,True
6,Sogrape Spain,None,None,Rejected,NaN,24/02/2026,29/04/2026,28/04/2026,hemos decidido no avanzar con la implementació...,Trinidad,trinidad.villegas@bodegaslan.com,None,None,True
7,Sogrape Spain,None,None,Rejected,NaN,24/02/2026,29/04/2026,28/04/2026,hemos decidido no avanzar con la implementació...,Julio,Julio.Martins@bodegaslan.com,None,None,False
8,Sogrape Spain,None,None,Rejected,NaN,24/02/2026,29/04/2026,28/04/2026,hemos decidido no avanzar con la implementació...,Alberto,a.saldon@bodegaslan.com,None,None,False
9,Sogrape Spain,None,None,Rejected,NaN,24/02/2026,29/04/2026,28/04/2026,hemos decidido no avanzar con la implementació...,Amaya,l.amatria@bodegaslan.com,None,None,False


In [45]:
# Ver distribución de sales_owner
df_a["sales_owner"].value_counts(dropna=False)

# Ver ejemplos donde se detectó owner
df_a[df_a["sales_owner"].notna()].head(10)

,brand_name,introduced_by,sales_owner,status,follow_up_later_until,first_contact_at,last_contact_at,last_response_at,notes,contact_full_name,contact_email,contact_phone,contact_role_title,contact_is_primary
21,Untamed Beverages,None,Juan,Rejected,NaN,6/04/2026,14/04/2026,14/04/2026,first contact - At this point it looks like we...,Joe,joe.colella@untamedbeverages.com,None,None,True
22,Untamed Beverages,None,Juan,Rejected,NaN,6/04/2026,14/04/2026,14/04/2026,first contact - At this point it looks like we...,Lee,lee.thaxton@untamedbeverages.com,None,None,False
23,Untamed Beverages,None,Juan,Rejected,NaN,6/04/2026,14/04/2026,14/04/2026,first contact - At this point it looks like we...,Bruce,bruce.carr@untamedbeverages.com,None,None,False
33,Crooked Tea,None,SofiG,Rejected,NaN,16/10/2025,23/03/2026,16/10/2025,first onboarding call and email,Brian,brianedelman@drinkcrooks.com,None,None,True
40,St Royale,None,SofiG,Rejected,NaN,30/09/2025,23/03/2026,10/10/2025,first onboarding call and email,Ted,tfason@stroyalevodka.com,None,None,True
41,St Royale,None,SofiG,Rejected,NaN,30/09/2025,23/03/2026,10/10/2025,first onboarding call and email,Team,socialmedia@stroyalevodka.com,None,None,False
53,Fidencio spirits,None,Juan,Rejected,NaN,8/09/2025,20/10/2025,9/09/2025,"said it was too expensive, wont respond to Jua...",Kristina,Kristina@fidenciospirits.com,None,None,True
69,Fizz,None,SofiG,Rejected,NaN,5/03/2026,7/04/2026,7/04/2026,first onboarding call and email. they want to ...,Julien,julien@ezmedcard.com,None,None,True
70,Fizz,None,SofiG,Rejected,NaN,5/03/2026,7/04/2026,7/04/2026,first onboarding call and email. they want to ...,Team,fizzsocialsoda@gmail.com,None,None,False
71,Fizz,None,SofiG,Rejected,NaN,5/03/2026,7/04/2026,7/04/2026,first onboarding call and email. they want to ...,Jason,jason@blackhogbrewing.com,None,None,False


In [46]:
df_a.to_csv("pipeline_a_v2.csv", index=False)